# xG model: single-hidden-layer neural network

Python translation of the original R Markdown analysis (`06_neural_network.html`).

The original analysis uses `tidyverse` and `nnet`. This notebook uses pandas, scikit-learn, matplotlib and seaborn. The neural network is implemented with `MLPClassifier` using one hidden layer and L-BFGS optimisation. Its `alpha` parameter provides L2 regularisation and is used here in place of `nnet`'s `decay`.

**Before running:** update `DATA_PATH` below so that it points to the cleaned shot-data CSV. The CSV is expected to contain the columns used in the original analysis, including `season`, `result`, `x`, `y` and `xg`.

## 0. Preparation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.utils import resample

RANDOM_STATE = 2026
DATA_PATH = Path("data/cleaned/shots_cleaned.csv")  # EDIT THIS PATH

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find the CSV at: {DATA_PATH.resolve()}\n"
        "Update DATA_PATH to the location of your cleaned shot data."
    )

shots = pd.read_csv(DATA_PATH)
print("Working directory:", Path.cwd())
print("Dataset shape:", shots.shape)
display(shots.head())

## 1. Data and features

In [ ]:
# Add a stable row identifier if one is not already present.
# This is retained for exporting predictions back to the group dataset.
if "row_id" not in shots.columns:
    shots["row_id"] = np.arange(len(shots))

# Goal location is assumed to be (120, 40), as in the R analysis.
shots["distance"] = np.sqrt((120 - shots["x"])**2 + (40 - shots["y"])**2)

drop_cols = ["xg", "result", "team", "season", "row_id"]
feature_cols = [c for c in shots.columns if c not in drop_cols]

continuous_cols = [
    "x", "y", "angle_to_goal", "distance", "minute",
    "team_goals_before", "opp_goals_before", "score_diff"
]
missing_required = [c for c in continuous_cols + ["season", "result", "xg"] if c not in shots.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Treat remaining non-numeric predictors as categorical and one-hot encode them.
categorical_cols = [c for c in feature_cols if c not in continuous_cols]
print("Shots:", len(shots), " Goal rate:", round(shots["result"].mean(), 4))
print("Features before one-hot encoding:", len(feature_cols))
print("Categorical features:", categorical_cols)

## 2. Train / validation / test split (by season)

In [ ]:
train_seasons = ["2021/22", "2022/23", "2023/24"]
val_seasons = ["2024/25"]
test_seasons = ["2025/26"]

train = shots[shots["season"].isin(train_seasons)].copy()
val = shots[shots["season"].isin(val_seasons)].copy()
test = shots[shots["season"].isin(test_seasons)].copy()

print({"train": len(train), "validation": len(val), "test": len(test)})

In [ ]:
def fit_preprocessor(df):
    """Fit scaling and one-hot encoding using this dataset only."""
    X = df[feature_cols].copy()
    for col in categorical_cols:
        X[col] = X[col].fillna("Missing").astype(str)
    for col in continuous_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].fillna(X[col].median())

    # One-hot encode categoricals; align future datasets to these columns.
    X = pd.get_dummies(X, columns=categorical_cols, dummy_na=False, dtype=float)
    scaler = StandardScaler()
    X[continuous_cols] = scaler.fit_transform(X[continuous_cols])
    return {"columns": X.columns.tolist(), "scaler": scaler, "categorical_cols": categorical_cols}

def transform_features(df, prep):
    X = df[feature_cols].copy()
    for col in prep["categorical_cols"]:
        X[col] = X[col].fillna("Missing").astype(str)
    for col in continuous_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].fillna(X[col].median())
    X = pd.get_dummies(X, columns=prep["categorical_cols"], dummy_na=False, dtype=float)
    X = X.reindex(columns=prep["columns"], fill_value=0)
    X[continuous_cols] = prep["scaler"].transform(X[continuous_cols])
    return X.to_numpy(dtype=float)

# Fit preprocessing on training data only; apply unchanged to validation data.
prep_train = fit_preprocessor(train)
X_train = transform_features(train, prep_train)
X_val = transform_features(val, prep_train)
y_train = train["result"].astype(int).to_numpy()
y_val = val["result"].astype(int).to_numpy()

## 3. Neural network model

The original R model uses a single hidden layer. `MLPClassifier` provides the equivalent basic architecture in scikit-learn. The `hidden_layer_sizes` argument sets the number of hidden neurons, `alpha` controls L2 regularisation, and `max_iter` sets the iteration limit.

The original `nnet` implementation uses entropy loss and BFGS optimisation. Scikit-learn's `MLPClassifier(solver="lbfgs")` also uses an L-BFGS optimiser, but the implementations are not identical. Consequently, the Python results should not be expected to reproduce the R results exactly.

In [ ]:
def fit_nn(X, y, size, decay, seeds=(1, 2, 3), maxit=300):
    models = []
    for seed in seeds:
        model = MLPClassifier(
            hidden_layer_sizes=(int(size),),
            activation="logistic",
            solver="lbfgs",
            alpha=float(decay),
            max_iter=maxit,
            random_state=seed
        )
        model.fit(X, y)
        models.append(model)
    return models

def predict_nn(models, X):
    probabilities = np.column_stack([m.predict_proba(X)[:, 1] for m in models])
    return probabilities.mean(axis=1)

### 3.1 Performance metrics

In [ ]:
def calculate_log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return log_loss(y, p, labels=[0, 1])

def calculate_brier(y, p):
    return brier_score_loss(y, p)

def calculate_auc(y, p):
    # ROC-AUC is undefined if the sample contains only one outcome class.
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

def evaluate(y, p, name):
    return pd.DataFrame([{
        "model": name,
        "log_loss": calculate_log_loss(y, p),
        "brier": calculate_brier(y, p),
        "auc": calculate_auc(y, p)
    }])

def calibration_data(y, p, name, bins=10):
    frame = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p)})
    frame["bin"] = pd.qcut(frame["p"], q=bins, labels=False, duplicates="drop")
    return (frame.groupby("bin", observed=True)
            .agg(mean_pred=("p", "mean"), obs_rate=("y", "mean"))
            .reset_index().assign(model=name))

## 4. Hyperparameter tuning on the validation season

In [ ]:
# Search over hidden-layer size and regularisation strength.
grid = pd.MultiIndex.from_product(
    [[2, 3, 5, 10], [0.01, 0.1, 0.3, 1.0]],
    names=["size", "decay"]
).to_frame(index=False)
grid["val_logloss"] = np.nan

for i, row in grid.iterrows():
    models = fit_nn(
        X_train, y_train,
        size=row["size"], decay=row["decay"],
        seeds=(1, 2), maxit=200
    )
    predictions = predict_nn(models, X_val)
    grid.loc[i, "val_logloss"] = calculate_log_loss(y_val, predictions)

display(grid.sort_values("val_logloss"))

In [ ]:
pivot = grid.pivot(index="decay", columns="size", values="val_logloss")
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="viridis_r")
plt.xlabel("Hidden units")
plt.ylabel("Weight decay (alpha)")
plt.title("Validation log loss by hyperparameter configuration")
plt.tight_layout()
plt.show()

best = grid.loc[grid["val_logloss"].idxmin()]
print("Selected configuration:")
display(best.to_frame("value"))

## 5. Final model and test-set evaluation

In [ ]:
# Combine training and validation data, then refit preprocessing.
trval = pd.concat([train, val], ignore_index=True)
prep_final = fit_preprocessor(trval)
X_final = transform_features(trval, prep_final)
X_test = transform_features(test, prep_final)
y_final = trval["result"].astype(int).to_numpy()
y_test = test["result"].astype(int).to_numpy()

final_models = fit_nn(
    X_final, y_final,
    size=int(best["size"]), decay=float(best["decay"]),
    seeds=(1, 2, 3, 4, 5), maxit=300
)

# sklearn exposes the number of iterations used; reaching max_iter is a warning sign.
convergence = pd.DataFrame({
    "network": np.arange(1, len(final_models) + 1),
    "iterations": [m.n_iter_ for m in final_models],
    "reached_max_iter": [m.n_iter_ >= 300 for m in final_models]
})
display(convergence)

p_nn = predict_nn(final_models, X_test)
results = pd.concat([
    evaluate(y_test, p_nn, "Neural network (Python)"),
    evaluate(y_test, test["xg"], "Understat xG"),
    evaluate(y_test, np.repeat(y_final.mean(), len(test)), "Constant base rate")
], ignore_index=True)
display(results)

In [ ]:
calibration = pd.concat([
    calibration_data(y_test, p_nn, "Neural network (Python)"),
    calibration_data(y_test, test["xg"], "Understat xG")
], ignore_index=True)

plt.figure(figsize=(7, 6))
sns.lineplot(data=calibration, x="mean_pred", y="obs_rate", hue="model", marker="o")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed goal rate")
plt.title("Calibration on the 2025/26 test season")
plt.tight_layout()
plt.show()

### 5.1 Exporting predictions for the group comparison

In [ ]:
# Convert row_id to zero-based indexing if the source uses one-based row numbers.
predictions_export = pd.DataFrame({
    "row_id": test["row_id"].to_numpy(),
    "season": test["season"].to_numpy(),
    "team": test["team"].to_numpy(),
    "nn_xg": p_nn,
    "goal": y_test,
    "understat_xg": test["xg"].to_numpy()
})
predictions_export.to_csv("nn_xg_predictions_2025_26.csv", index=False)
display(predictions_export.head())

### 5.2 Examining the model's xG predictions

In [ ]:
xg_out = test[["row_id", "team", "season", "minute", "x", "y"]].copy()
xg_out["goal"] = y_test
xg_out["nn_xg"] = p_nn
xg_out["understat_xg"] = test["xg"].to_numpy()

display(xg_out.nlargest(10, "nn_xg"))
display(xg_out.sample(10, random_state=RANDOM_STATE))

team_comparison = (xg_out.groupby("team")
    .agg(shots=("goal", "size"),
         goals=("goal", "sum"),
         nn_xg_total=("nn_xg", "sum"),
         understat_xg_total=("understat_xg", "sum"))
    .sort_values("nn_xg_total", ascending=False))
display(team_comparison)

## 6. Evaluation without penalties and by shot type

In [ ]:
if "is_penalty" not in test.columns:
    raise KeyError("The test data must contain is_penalty for this analysis.")

no_penalty = test["is_penalty"].to_numpy() == 0
penalty_comparison = pd.concat([
    evaluate(y_test, p_nn, "Neural network - all shots"),
    evaluate(y_test[no_penalty], p_nn[no_penalty], "Neural network - excluding penalties"),
    evaluate(y_test, test["xg"], "Understat xG - all shots"),
    evaluate(y_test[no_penalty], test["xg"].to_numpy()[no_penalty], "Understat xG - excluding penalties")
], ignore_index=True)
display(penalty_comparison)

In [ ]:
cal_no_pen = pd.concat([
    calibration_data(y_test[no_penalty], p_nn[no_penalty], "Neural network (no penalties)"),
    calibration_data(y_test[no_penalty], test["xg"].to_numpy()[no_penalty], "Understat xG (no penalties)")
], ignore_index=True)

plt.figure(figsize=(7, 6))
sns.lineplot(data=cal_no_pen, x="mean_pred", y="obs_rate", hue="model", marker="o")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed goal rate")
plt.title("Calibration on the test season, excluding penalties")
plt.tight_layout()
plt.show()

In [ ]:
required_type_cols = ["is_penalty", "is_set_piece", "is_open_play"]
missing = [c for c in required_type_cols if c not in test.columns]
if missing:
    raise KeyError(f"Missing shot-type columns: {missing}")

shot_type = np.select(
    [test["is_penalty"].eq(1), test["is_set_piece"].eq(1), test["is_open_play"].eq(1)],
    ["Penalty", "Set piece", "Open play"],
    default="Other"
)

type_results = []
for shot in pd.unique(shot_type):
    idx = shot_type == shot
    part = pd.concat([
        evaluate(y_test[idx], p_nn[idx], "Neural network"),
        evaluate(y_test[idx], test["xg"].to_numpy()[idx], "Understat xG")
    ], ignore_index=True)
    part.insert(0, "shot_type", shot)
    part.insert(1, "shots", int(idx.sum()))
    part.insert(2, "goals", int(y_test[idx].sum()))
    type_results.append(part)

display(pd.concat(type_results, ignore_index=True).sort_values(["shot_type", "model"]))

## 7. Uncertainty in the comparison with Understat

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
B = 500
n = len(test)

# Resample whole matches where a match identifier is available.
cluster_candidates = ["match_id", "game_id", "fixture_id"]
cluster_col = next((c for c in cluster_candidates if c in test.columns), None)

if cluster_col is not None:
    clusters = test[cluster_col].to_numpy()
    unique_clusters = pd.unique(clusters)
    cluster_indices = [np.flatnonzero(clusters == c) for c in unique_clusters]
    print(f"Resampling matches using {cluster_col}")
else:
    print("No match identifier found: resampling individual shots.")

def metric_differences(indices):
    y = y_test[indices]
    return np.array([
        calculate_log_loss(y, p_nn[indices]) - calculate_log_loss(y, test["xg"].to_numpy()[indices]),
        calculate_brier(y, p_nn[indices]) - calculate_brier(y, test["xg"].to_numpy()[indices]),
        calculate_auc(y, p_nn[indices]) - calculate_auc(y, test["xg"].to_numpy()[indices])
    ])

observed = metric_differences(np.arange(n))
boot = np.empty((B, 3))

for b in range(B):
    if cluster_col is None:
        idx = rng.integers(0, n, size=n)
    else:
        sampled = rng.integers(0, len(cluster_indices), size=len(cluster_indices))
        idx = np.concatenate([cluster_indices[j] for j in sampled])
    boot[b] = metric_differences(idx)

bootstrap_results = pd.DataFrame({
    "metric": ["log_loss", "brier", "auc"],
    "difference": observed,
    "ci_low": np.nanquantile(boot, 0.025, axis=0),
    "ci_high": np.nanquantile(boot, 0.975, axis=0)
})
display(bootstrap_results)

## 8. Exploration for the performance-metric part of the brief

### 8a. Generalising to unseen shot types

In [ ]:
# Train only on open-play shots and evaluate on set-piece shots.
op_train = trval[trval["is_open_play"].eq(1)].copy()
sp_test = test[test["is_set_piece"].eq(1)].copy()

prep_open = fit_preprocessor(op_train)
X_open_train = transform_features(op_train, prep_open)
X_set_piece = transform_features(sp_test, prep_open)
X_set_piece_all = transform_features(sp_test, prep_final)

open_models = fit_nn(
    X_open_train, op_train["result"].astype(int).to_numpy(),
    size=int(best["size"]), decay=float(best["decay"]),
    seeds=(1, 2, 3, 4, 5), maxit=300
)
p_open_only = predict_nn(open_models, X_set_piece)
p_all_shots = predict_nn(final_models, X_set_piece_all)
y_set_piece = sp_test["result"].astype(int).to_numpy()

display(pd.concat([
    evaluate(y_set_piece, p_open_only, "Trained on open play only"),
    evaluate(y_set_piece, p_all_shots, "Trained on all shots"),
    evaluate(y_set_piece, sp_test["xg"], "Understat xG")
], ignore_index=True))

### 8b. Permutation importance

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_repeats = 10
base_log_loss = calculate_log_loss(y_test, p_nn)
importance = {}

for j, feature in enumerate(prep_final["columns"]):
    increases = []
    for _ in range(n_repeats):
        X_permuted = X_test.copy()
        X_permuted[:, j] = rng.permutation(X_permuted[:, j])
        p_permuted = predict_nn(final_models, X_permuted)
        increases.append(calculate_log_loss(y_test, p_permuted) - base_log_loss)
    importance[feature] = np.mean(increases)

importance_df = (pd.Series(importance, name="log_loss_increase")
                 .sort_values(ascending=False).head(15).reset_index())
importance_df.columns = ["feature", "log_loss_increase"]
display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="log_loss_increase", y="feature")
plt.xlabel("Increase in test log loss")
plt.ylabel("")
plt.title("Permutation importance (top 15)")
plt.tight_layout()
plt.show()

## 9. Conditions for a neural network to work

The network is intended to learn non-linear relationships between shot characteristics and the probability of scoring. Its performance depends on having sufficient, relevant training data; handling missing values; encoding categorical variables numerically; and scaling continuous variables. The chronological split provides an assessment on a later season, although changes in shot distributions or playing styles may affect generalisation.

## 10. Advantages and disadvantages of the neural network

The main advantage is the ability to represent non-linear relationships and interactions between features. However, the learned weights are difficult to interpret, and the model requires choices about architecture, regularisation and random initialisation. Greater flexibility does not necessarily imply better predictive performance, so the model should be compared with the other approaches on the same test data.

## 11. Potential extensions

Possible extensions include fitting separate models for different shot types, comparing feature importance consistently across models, and investigating calibration across ranges of predicted xG. Grouped permutation importance may be useful because coordinates, distance and angle are correlated.

---

### Important implementation notes

- This is a **Python reimplementation**, not a line-for-line conversion. `MLPClassifier` and R's `nnet` differ, so the numerical results will differ.
- The original HTML reports that four of five R networks reached the iteration limit. Check the `convergence` table above; if models reach `max_iter`, increase it and refit.
- Confirm that the cleaned CSV has the expected columns and that `row_id` uses the same row convention as the group notebook. The export cell currently preserves the CSV's `row_id`; subtract one only if your source IDs are one-based and the group notebook expects zero-based IDs.
- The notebook uses one-hot encoding for categorical predictors. This is a deliberate Python-side preprocessing step; check the resulting feature columns before comparing against the R model.